In [0]:
from pyspark.sql import functions as F


def add_ingestion_medatat(df):
    final_df = df.withColumn("Ingestion_Timestamp", F.current_timestamp()).withColumn(
        "Source_File", F.col("_metadata.file_path")
    )
    return final_df

In [0]:
def write_to_bronze(input_df, table, batch_id):

    final_df = input_df.withColumn("batch_id", F.lit(batch_id))

    (
        final_df.write.format("delta")
        .mode("overwrite")
        .partitionBy("batch_id")
        .option("replaceWhere", f"batch_id = '{batch_id}'")
        .saveAsTable(table)
    )

In [0]:
from delta.tables import DeltaTable


def write_to_silver(input_df, target_table, merge_condidtion, columns_to_update):
    """
    Creates the Delta table if it does not exist.
    Otherwise merges the input DataFrame into the target table.
    """
    final_df = input_df.withColumn(
        "created_timestamp", F.current_timestamp()
    ).withColumn("updated_timestamp", F.current_timestamp())

    if not spark.catalog.tableExists(target_table):
        (final_df.write.format("delta").mode("overwrite").saveAsTable(target_table))
    else:
        target = DeltaTable.forName(spark, target_table)

        update_map = {column: f"s.{column}" for column in columns_to_update}
        update_map["updated_timestamp"] = "s.updated_timestamp"
        
        (
            target.alias("t")
            .merge(
                final_df.alias("s"), 
                merge_condidtion        
            )
            .whenMatchedUpdate(
                condition=merge_condidtion,
                set=update_map
            )
            .whenNotMatchedInsertAll()
            .execute()
        )